# Libs

In [15]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
# try lib polars
# import polars as pl

In [16]:
'''  
Pré-processamento sugerido:

Remoção de valores ambíguos como "Don’t know", "Refused to answer".

Eliminação de colunas com mais de 30% de valores faltantes.

Substituição de códigos como 88/888 por zero em variáveis de contagem de dias.

Aplicação de normalização MinMaxScaler.

Redução de colinearidade com análise de correlação (Pearson).

Binning em variáveis contínuas como altura e peso.

'''

'  \nPré-processamento sugerido:\n\nRemoção de valores ambíguos como "Don’t know", "Refused to answer".\n\nEliminação de colunas com mais de 30% de valores faltantes.\n\nSubstituição de códigos como 88/888 por zero em variáveis de contagem de dias.\n\nAplicação de normalização MinMaxScaler.\n\nRedução de colinearidade com análise de correlação (Pearson).\n\nBinning em variáveis contínuas como altura e peso.\n\n'

In [17]:
# mapeando o diretório do projeto e do arquivo notebook 
diretorio_atual_projeto = os.getcwd() # Diretório atual do arquivo
notebook_dir_project_predict = os.path.normpath(f"{diretorio_atual_projeto}{os.sep}..{os.sep}..{os.sep}") + os.sep # Diretório do projeto
print(diretorio_atual_projeto)
print(notebook_dir_project_predict)

/home/ed/lgcm/projects/riskPredictionDeseases/dev/notebooks
/home/ed/lgcm/projects/riskPredictionDeseases/


# Functions

In [ ]:
# mapear o DataFrame de acordo com o codebook 

def mapear_colunas_para_rotulo(df_brfss, codebook_df, sufixo='_map'):
    """
    Substitui colunas do DataFrame por versões mapeadas com rótulos do codebook.
    A coluna original é excluída e substituída por uma nova com sufixo (default: _map).
    Se o valor não for mapeável, mantém o valor original.

    Parâmetros:
        df_brfss (pd.DataFrame): dados originais
        codebook_df (pd.DataFrame): DataFrame do codebook com 'SAS Variable Name', 'Value', 'Value Label'
        sufixo (str): sufixo para a nova coluna (default: '_map')

    Retorno:
        pd.DataFrame com colunas mapeadas
    """
    def limpar_valor(v):
        if pd.isna(v):
            return "BLANK"
        try:
            return str(int(v))  # converte 1.0 → '1'
        except:
            return str(v).strip()

    df_resultado = df_brfss.copy()
    colunas_mapeadas = {}

    with tqdm(total=len(df_brfss.columns), desc="Processing columns") as pbar:

        for coluna in df_brfss.columns:
            pbar.update(1)
            try:
                # Extrai mapeamentos apenas para a variável atual
                codebook_var = codebook_df[codebook_df['SAS Variable Name'] == coluna]
                codebook_var = codebook_var.dropna(subset=['Value'])

                if codebook_var.empty:
                    continue

                # Cria o dicionário de mapeamento
                mapa_valores = dict(zip(
                    codebook_var['Value'].astype(str).str.strip(),
                    codebook_var['Value Label']
                ))

                if not mapa_valores:
                    continue

                # Aplica mapeamento apenas onde existir valor no dicionário
                serie_convertida = df_brfss[coluna].apply(limpar_valor)
                serie_mapeada = serie_convertida.apply(lambda x: mapa_valores.get(x, x))  # mantém valor original se não estiver no dicionário
                colunas_mapeadas[coluna + sufixo] = serie_mapeada
            
            except Exception as e:
                print(f"Erro ao mapear coluna '{coluna}': {e}")
                continue

            

    # Cria DataFrame com as colunas mapeadas
    df_mapeadas = pd.DataFrame(colunas_mapeadas)

    # Remove colunas originais que foram mapeadas
    colunas_para_remover = [col.replace(sufixo, '') for col in tqdm(df_mapeadas.columns, desc='Remove suport columns')]
    df_resultado = df_resultado.drop(columns=colunas_para_remover)

    # Junta com colunas mapeadas
    df_resultado = pd.concat([df_mapeadas, df_resultado], axis=1)

    return df_resultado

# lista de colunas onde devemos retirar da string os seguintes caracteres "b'" di começo e "'" no final # IDATE IMONTH IDAY IYEAR
# IDATE IMONTH IDAY IYEAR
def limpar_colunas_data(df):
    """
    Função para limpar as colunas de data do DataFrame
    """
    # Limpando as colunas
    df['IDATE'] = df['IDATE'].str.replace("b'", "").str.replace("'", "")
    df['IMONTH'] = df['IMONTH'].str.replace("b'", "").str.replace("'", "")
    df['SEQNO'] = df['SEQNO'].str.replace("b'", "").str.replace("'", "")
    # df['IDAY'] = df['IDAY'].str.replace("b'", "").str.replace("'", "")
    # df['IYEAR'] = df['IYEAR'].str.replace("b'", "").str.replace("'", "")
    
    return df

# faremos uma lista de colunas que serão ignoradas pois todos os dados são o mesmo valor ou nulo e são dados de identificação de amostra





In [19]:
def read_data_and_codebook(path_csv_data, path_csv_codebook):
    """
    Função para ler os dados e o código  .
    """
    # Lendo os dados
    df = pd.read_csv(path_csv_data)
    
    # Lendo o código
    codebook = pd.read_csv(path_csv_codebook)
    
    return df, codebook

# transformar os dados categorizados como dont know ou refused em NaN
def transform_dont_know_refused_to_nan(df):
    """
    Função para transformar os dados categorizados como dont know ou refused em NaN
    """
    # Transformando os dados categorizados como dont know ou refused em NaN
    df = df.replace({'dont know': np.nan, 'refused': np.nan})
    
    return df

# retirar as colunas com mais de 30% de missing data conforme artigo
def remove_columns_with_missing_data(df, threshold=0.3):
    """
    Função para remover colunas com mais de 30% de missing data
    """
    # Calculando o percentual de missing data
    missing_data = df.isnull().mean()
    
    # Removendo as colunas com mais de 30% de missing data
    df = df.loc[:, missing_data < threshold]
    
    return df

In [20]:
# implementada a substituição por vazio 
def substituir_valores_por_zero_baseado_no_codebook(df_brfss, codebook_df, strings_para_zero, sufixo='_processed'):
    """
    Processa colunas do DataFrame:
    Se o rótulo de um valor no codebook contiver alguma das 'strings_para_zero',
    o valor correspondente no DataFrame de dados é substituído por 0.
    Caso contrário, o valor original no DataFrame de dados é mantido.
    As colunas processadas substituem as originais com um sufixo.

    Parâmetros:
        df_brfss (pd.DataFrame): DataFrame original com os dados.
        codebook_df (pd.DataFrame): DataFrame do codebook com 'SAS Variable Name', 'Value', 'Value Label'.
        strings_para_zero (list): Lista de strings que, se encontradas no 'Value Label'
                                  do codebook, farão com que o valor original no df_brfss
                                  seja substituído por 0.
        sufixo (str): Sufixo para as novas colunas processadas.

    Retorno:
        pd.DataFrame com as colunas processadas.
    """

    def limpar_valor_para_lookup(v):
        """Limpa e converte valor para string para lookup no dicionário do codebook."""
        if pd.isna(v):
            return "INTERNAL_NAN_REPR" # Representação interna para NaNs originais dos dados
        try:
            # Tenta converter para int (para lidar com 1.0 -> '1'), depois para string
            return str(int(float(v)))
        except ValueError:
            # Se não puder ser convertido para float/int, usa como string
            return str(v).strip()
        except Exception:
            return str(v).strip() # Fallback

    df_processado = df_brfss.copy()
    colunas_originais_para_remover = []

    strings_para_zero_lower = [s.lower() for s in strings_para_zero]

    with tqdm(total=len(df_processado.columns), desc="Processando colunas") as pbar:
        for coluna in df_processado.columns:
            pbar.update(1)
            
            # Pega as entradas do codebook para a coluna atual
            codebook_var_atual = codebook_df[codebook_df['SAS Variable Name'] == coluna]
            
            if codebook_var_atual.empty:
                continue # Pula para a próxima coluna se não houver info no codebook

            # Cria um mapa de código (Value) para rótulo (Value Label)
            # Limpa os 'Value' do codebook para string para consistência
            mapa_codigo_rotulo = dict(zip(
                codebook_var_atual['Value'].apply(limpar_valor_para_lookup),
                codebook_var_atual['Value Label']
            ))
            
            # Série original da coluna a ser processada
            serie_original = df_processado[coluna].copy()
            # Série que será modificada (começa como uma cópia)
            serie_modificada = df_processado[coluna].copy()

            for idx, valor_original_na_serie in serie_original.items():
                valor_limpo_dados = limpar_valor_para_lookup(valor_original_na_serie)
                
                # Pega o rótulo do codebook para o valor limpo dos dados
                rotulo_do_codebook = mapa_codigo_rotulo.get(valor_limpo_dados)

                if rotulo_do_codebook: # Se encontrou um rótulo no codebook
                    # Verifica se alguma das strings_para_zero está no rótulo
                    if any(s_lower in str(rotulo_do_codebook).lower() for s_lower in strings_para_zero_lower):
                        serie_modificada.loc[idx] = np.nan # 0
                    # else: o valor original já está em serie_modificada, então não faz nada
                elif valor_limpo_dados == "INTERNAL_NAN_REPR" and "blank" in strings_para_zero_lower:
                    # Trata NaNs originais nos dados se "blank" for uma string para zerar
                    serie_modificada.loc[idx] = np.nan # 0
                # else: valor não encontrado no codebook ou rótulo não corresponde, mantém original

            # Atualiza a coluna no DataFrame processado
            df_processado[coluna + sufixo] = serie_modificada
            if sufixo : # Adiciona à lista para remover depois, apenas se houver sufixo
                colunas_originais_para_remover.append(coluna)
    
    # Remove as colunas originais que foram processadas (se o sufixo for diferente de vazio)
    if sufixo and colunas_originais_para_remover:
        colunas_existentes_para_remover = [col for col in colunas_originais_para_remover if col in df_processado.columns]
        df_processado.drop(columns=colunas_existentes_para_remover, inplace=True)
        
    return df_processado

In [21]:
# remoção de strings indesejadas

def limpar_valores_indesejados_codebook(df_brfss, codebook_df, rotulos_invalidos):
    """
    Substitui por NaN os valores do DataFrame que correspondem a rótulos inválidos do codebook.

    Parâmetros:
        df_brfss (pd.DataFrame): dados originais com valores numéricos
        codebook_df (pd.DataFrame): codebook com colunas 'SAS Variable Name', 'Value', 'Value Label'
        rotulos_invalidos (list): lista de strings com rótulos que devem ser tratados como NaN

    Retorno:
        pd.DataFrame com valores substituídos por NaN onde os rótulos são inválidos
    """
    df_resultado = df_brfss.copy()

    for coluna in df_resultado.columns:
        try:
            codebook_var = codebook_df[codebook_df['SAS Variable Name'] == coluna]
            codebook_var = codebook_var.dropna(subset=['Value', 'Value Label'])

            # Filtra os valores que têm rótulos indesejados
            valores_invalidos = codebook_var[
                codebook_var['Value Label'].str.strip().isin(rotulos_invalidos)
            ]['Value']

            # Converte para float para comparar com os dados
            valores_invalidos_float = valores_invalidos.astype(float).tolist()

            # Substitui no dado
            df_resultado[coluna] = df_resultado[coluna].apply(
                lambda x: np.nan if x in valores_invalidos_float else x
            )

        except Exception as e:
            print(f"Erro ao processar coluna '{coluna}': {e}")
            continue

    return df_resultado


In [22]:
# retirar as linhas onde a coluna MICHD for nulo

def remover_linhas_com_alvo_nulo(df, nome_coluna_alvo):
    """
    Remove linhas de um DataFrame onde a coluna alvo especificada é nula (NaN).

    Parâmetros:
        df (pd.DataFrame): DataFrame de entrada.
        nome_coluna_alvo (str): Nome da coluna alvo para verificar valores nulos.

    Retorno:
        pd.DataFrame: DataFrame com as linhas nulas na coluna alvo removidas.
    """
    if nome_coluna_alvo not in df.columns:
        print(f"Erro: A coluna '{nome_coluna_alvo}' não existe no DataFrame.")
        return df # Retorna o DataFrame original se a coluna não existir

    linhas_antes = len(df)
    df_processado = df.dropna(subset=[nome_coluna_alvo])
    linhas_depois = len(df_processado)
    
    print(f"Coluna alvo para remoção de nulos: '{nome_coluna_alvo}'")
    print(f"Linhas antes da remoção: {linhas_antes}")
    print(f"Linhas removidas: {linhas_antes - linhas_depois}")
    print(f"Linhas após a remoção: {linhas_depois}")
    
    return df_processado

# Scripts

Ler o codebook e o respectivo dado

In [ ]:
'''       
No codebook,
SAS Variable Name - Coluna do Dado bruto 
Value - Valor no dado bruto
Value Label - Descrição do valor no dado bruto

'''
# precisaremos implementar a metodologia descrita no artigo gerando assim um primeiro dataset para treinamento
# ler o csv do dado bruto e do codebook para efetuar as transformações
raw_data_brfss = os.path.normpath(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}brfss_2023.csv")
codebook_file = os.path.normpath(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023_variaveis_expandidas_translated.csv")

df_2023 , codebook_2023 = read_data_and_codebook(raw_data_brfss, codebook_file)
# df_2023

In [6]:
codebook_2023

,Label,Section Name,Section Number,Question Number,Column,Type of Variable,SAS Variable Name,Question Prologue,Question,Value,Value Label,Frequency,Percentage,Weighted Percentage,label_translate,section_name_translate,question_translate,value_label_translate
0,State FIPS Code,Record Identification,0.0,1,1-2,Num,_STATE,NaN,State FIPS Code,1,Alabama,"4,362",1.01,1.58,Código de FIPS do estado,Identificação de registro,Código de FIPS do estado,Alabama
1,State FIPS Code,Record Identification,0.0,1,1-2,Num,_STATE,NaN,State FIPS Code,2,Alaska,"5,525",1.28,0.22,Código de FIPS do estado,Identificação de registro,Código de FIPS do estado,Alasca
2,State FIPS Code,Record Identification,0.0,1,1-2,Num,_STATE,NaN,State FIPS Code,4,Arizona,"12,036",2.78,2.31,Código de FIPS do estado,Identificação de registro,Código de FIPS do estado,Arizona
3,State FIPS Code,Record Identification,0.0,1,1-2,Num,_STATE,NaN,State FIPS Code,5,Arkansas,"5,351",1.23,0.94,Código de FIPS do estado,Identificação de registro,Código de FIPS do estado,Arkansas
4,State FIPS Code,Record Identification,0.0,1,1-2,Num,_STATE,NaN,State FIPS Code,6,California,"11,976",2.76,12.18,Código de FIPS do estado,Identificação de registro,Código de FIPS do estado,Califórnia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1891,Always Wear Seat Belts,Calculated Variables,NaN,2,2109,Num,_RFSEAT3,NaN,Always Wear Seat Belts Calculated Variable,2,Don't Always Wear Seat Belt,"50,738",11.71,10.99,Sempre use cintos de segurança,Variáveis ​​calculadas,Sempre use cintos de segurança variável calculada,Nem sempre use cinto de segurança
1892,Always Wear Seat Belts,Calculated Variables,NaN,2,2109,Num,_RFSEAT3,NaN,Always Wear Seat Belts Calculated Variable,9,Don't know/Not Sure Or Refused/Missing,"31,505",7.27,8.70,Sempre use cintos de segurança,Variáveis ​​calculadas,Sempre use cintos de segurança variável calculada,Não sei/não tenho certeza ou recusou/ausente
1893,Drinking and Driving,Calculated Variables,NaN,3,2110,Num,_DRNKDRV,NaN,Drinking and Driving (Reported having driven ...,1,Have driven after having too much to drink,"6,192",1.43,1.36,Bebendo e dirigindo,Variáveis ​​calculadas,Beber e dirigir (relatou ter dirigido pelo men...,Dirigiram depois de beber muito
1894,Drinking and Driving,Calculated Variables,NaN,3,2110,Num,_DRNKDRV,NaN,Drinking and Driving (Reported having driven ...,2,Have not driven after having too much to drink,"205,541",47.43,45.81,Bebendo e dirigindo,Variáveis ​​calculadas,Beber e dirigir (relatou ter dirigido pelo men...,Não dirigiram depois de beber muito


In [ ]:
# detecção das colunas que não estão no codebook
colunas_nao_mapeadas = [col for col in df_2023.columns if col not in codebook_2023['SAS Variable Name'].values]
print("Colunas não mapeadas no codebook:")

df_2023_removed_columns = df_2023.drop(columns=colunas_nao_mapeadas, errors='ignore')
# df_2023_removed_columns

# deletar colunas , existentes no codebook e dado mas inúteis para a análize



# identificar se nas colunas restantes ignorando o valores vazios tem algum valor não representado no codebook e listar essas colunas e o valor presente no dado que não está no codebook
def encontrar_e_salvar_valores_nao_mapeados(df_brfss, codebook_df, caminho_arquivo_saida):
    """
    Identifica, para cada coluna do DataFrame, os valores que não possuem
    um 'Value Label' correspondente no codebook. Valores nulos (NaN)
    no DataFrame de dados são ignorados. Os resultados são salvos em um arquivo de texto.

    Parâmetros:
        df_brfss (pd.DataFrame): DataFrame original com os dados.
        codebook_df (pd.DataFrame): DataFrame do codebook com 'SAS Variable Name',
                                    'Value', e 'Value Label'.
        caminho_arquivo_saida (str): Caminho completo para o arquivo .txt onde os
                                     resultados serão salvos.

    Retorno:
        dict: Um dicionário onde as chaves são nomes de colunas e os valores
              são listas de valores únicos daquela coluna que não foram
              encontrados nos códigos ('Value') do codebook para aquela variável.
              Retorna apenas colunas que tiveram valores não mapeados.
              Retorna um dicionário vazio se nenhum valor não mapeado for encontrado.
    """

    def limpar_valor_para_comparacao(v):
        """Limpa e converte valor para string para comparação com os códigos do codebook."""
        if pd.isna(v): # Se o valor já for NaN, não há como limpar para string de forma útil aqui.
            return None # Será filtrado depois pelo dropna() nos valores únicos da coluna.
        try:
            return str(int(float(v)))
        except ValueError:
            return str(v).strip()
        except Exception:
            return str(v).strip()

    valores_nao_encontrados_geral = {}

    with tqdm(total=len(df_brfss.columns), desc="Verificando colunas") as pbar:
        for coluna in df_brfss.columns:
            pbar.update(1)
            
            codebook_var_atual = codebook_df[codebook_df['SAS Variable Name'] == coluna]
            
            if codebook_var_atual.empty:
                continue

            codigos_no_codebook_para_coluna = set(
                codebook_var_atual['Value'].dropna().apply(limpar_valor_para_comparacao)
            )

            if not codigos_no_codebook_para_coluna:
                continue
                
            # Pega os valores únicos da coluna no DataFrame de dados, IGNORANDO NaNs
            valores_unicos_na_coluna_dados = df_brfss[coluna].dropna().unique()
            
            valores_nao_encontrados_nesta_coluna = set()

            for valor_dados in valores_unicos_na_coluna_dados:
                valor_dados_limpo = limpar_valor_para_comparacao(valor_dados)
                
                # Se valor_dados_limpo for None (era NaN originalmente), não deve ser comparado
                if valor_dados_limpo is None:
                    continue

                if valor_dados_limpo not in codigos_no_codebook_para_coluna:
                    valores_nao_encontrados_nesta_coluna.add(valor_dados) 
            
            if valores_nao_encontrados_nesta_coluna:
                # Converte para lista e ordena para consistência na saída
                # Garante que os valores sejam strings para evitar problemas de tipo misto no sort
                valores_nao_encontrados_geral[coluna] = sorted(list(map(str, valores_nao_encontrados_nesta_coluna)))
    
    # Salvar os resultados no arquivo de texto
    try:
        with open(caminho_arquivo_saida, 'w', encoding='utf-8') as f:
            if valores_nao_encontrados_geral:
                f.write("Valores não encontrados no codebook (ignorando NaNs nos dados):\n")
                f.write("="*60 + "\n")
                for coluna, valores in valores_nao_encontrados_geral.items():
                    f.write(f"Coluna: {coluna}\n")
                    f.write(f"Valores não mapeados: {valores}\n")
                    f.write("-" * 40 + "\n")
                print(f"\nResultados salvos em: {caminho_arquivo_saida}")
            else:
                f.write("Nenhum valor não mapeado encontrado (ignorando NaNs nos dados).\n")
                print(f"\nNenhum valor não mapeado encontrado. Arquivo salvo em: {caminho_arquivo_saida}")
    except IOError as e:
        print(f"Erro ao salvar o arquivo em '{caminho_arquivo_saida}': {e}")
        # Retorna o dicionário mesmo se o salvamento falhar, para não perder os dados
        return valores_nao_encontrados_geral
    except Exception as e:
        print(f"Ocorreu um erro inesperado ao tentar salvar o arquivo: {e}")
        return valores_nao_encontrados_geral
                
    return valores_nao_encontrados_geral
caminho_arquivo_saida = os.path.normpath(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}valores_nao_mapeados.txt")
valores_nao_mapeados = encontrar_e_salvar_valores_nao_mapeados(df_2023_removed_columns, codebook_2023, caminho_arquivo_saida)


'''def encontrar_valores_nao_mapeados_no_codebook(df_brfss, codebook_df):
    """
    Identifica, para cada coluna do DataFrame, os valores que não possuem
    um 'Value Label' correspondente no codebook. Valores nulos (NaN)
    no DataFrame de dados são ignorados.

    Parâmetros:
        df_brfss (pd.DataFrame): DataFrame original com os dados.
        codebook_df (pd.DataFrame): DataFrame do codebook com 'SAS Variable Name',
                                    'Value', e 'Value Label'.

    Retorno:
        dict: Um dicionário onde as chaves são nomes de colunas e os valores
              são listas de valores únicos daquela coluna que não foram
              encontrados nos códigos ('Value') do codebook para aquela variável.
              Retorna apenas colunas que tiveram valores não mapeados.
    """

    def limpar_valor_para_comparacao(v):
        """Limpa e converte valor para string para comparação com os códigos do codebook."""
        # Não trata pd.isna(v) aqui, pois será filtrado antes
        try:
            # Tenta converter para int (para lidar com 1.0 -> '1'), depois para string
            return str(int(float(v)))
        except ValueError:
            # Se não puder ser convertido para float/int, usa como string
            return str(v).strip()
        except Exception:
            return str(v).strip() # Fallback

    valores_nao_encontrados_geral = {}

    with tqdm(total=len(df_brfss.columns), desc="Verificando colunas") as pbar:
        for coluna in df_brfss.columns:
            pbar.update(1)
            
            # 1. Pega os códigos ('Value') definidos no codebook para a coluna atual
            codebook_var_atual = codebook_df[codebook_df['SAS Variable Name'] == coluna]
            
            if codebook_var_atual.empty:
                # Se a variável nem existe no codebook, todos os seus valores (exceto NaN) são "não mapeados"
                # No entanto, a lógica original da sua função de mapeamento pularia essa coluna.
                # Vamos considerar aqui que, se não há entrada no codebook, não há como mapear.
                # print(f"Aviso: Coluna '{coluna}' não encontrada no codebook.")
                continue

            # Limpa e pega os códigos ('Value') do codebook como uma lista de strings
            codigos_no_codebook_para_coluna = set(
                codebook_var_atual['Value'].dropna().apply(limpar_valor_para_comparacao)
            )

            if not codigos_no_codebook_para_coluna:
                # Se o codebook tem a variável, mas não tem 'Values' definidos para ela
                continue
                
            # 2. Pega os valores únicos da coluna no DataFrame de dados, ignorando NaNs
            valores_unicos_na_coluna_dados = df_brfss[coluna].dropna().unique()
            
            valores_nao_encontrados_nesta_coluna = set()

            for valor_dados in valores_unicos_na_coluna_dados:
                valor_dados_limpo = limpar_valor_para_comparacao(valor_dados)
                
                # 3. Verifica se o valor limpo dos dados está nos códigos do codebook
                if valor_dados_limpo not in codigos_no_codebook_para_coluna:
                    valores_nao_encontrados_nesta_coluna.add(valor_dados) # Adiciona o valor original
            
            if valores_nao_encontrados_nesta_coluna:
                valores_nao_encontrados_geral[coluna] = sorted(list(valores_nao_encontrados_nesta_coluna))
                
    return valores_nao_encontrados_geral

valores_nao_mapeados = encontrar_valores_nao_mapeados_no_codebook(df_2023_removed_columns, codebook_2023)
print("Valores não mapeados encontrados:")
for coluna, valores in valores_nao_mapeados.items():
    print(f"{coluna}: {valores}")


'''
# limpar as colunas de data

# se não tiver mudar no dado para vazio



Colunas não mapeadas no codebook:


Verificando colunas: 100%|██████████| 343/343 [00:01<00:00, 304.89it/s]



Resultados salvos em: /home/ed/lgcm/projects/riskPredictionDeseases/data/intermediate/2023/valores_nao_mapeados.txt


'def encontrar_valores_nao_mapeados_no_codebook(df_brfss, codebook_df):\n    """\n    Identifica, para cada coluna do DataFrame, os valores que não possuem\n    um \'Value Label\' correspondente no codebook. Valores nulos (NaN)\n    no DataFrame de dados são ignorados.\n\n    Parâmetros:\n        df_brfss (pd.DataFrame): DataFrame original com os dados.\n        codebook_df (pd.DataFrame): DataFrame do codebook com \'SAS Variable Name\',\n                                    \'Value\', e \'Value Label\'.\n\n    Retorno:\n        dict: Um dicionário onde as chaves são nomes de colunas e os valores\n              são listas de valores únicos daquela coluna que não foram\n              encontrados nos códigos (\'Value\') do codebook para aquela variável.\n              Retorna apenas colunas que tiveram valores não mapeados.\n    """\n\n    def limpar_valor_para_comparacao(v):\n        """Limpa e converte valor para string para comparação com os códigos do codebook."""\n        # Não trata

## 1 Dado bruto mapeado para exploração

Mapear codebook e gerar um novo dataset

In [7]:
df_2023_to_treat = df_2023.copy()
# df_cleaned_date = limpar_colunas_data(df_2023_to_treat)

df_map = mapear_colunas_para_rotulo(df_2023_to_treat, codebook_2023)
df_map.to_csv(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023_maped.csv")
df_map

Remove suport columns: 100%|██████████| 343/343 [00:00<00:00, 2573606.93it/s]


,_STATE_map,FMONTH_map,IDATE_map,IMONTH_map,IDAY_map,IYEAR_map,DISPCODE_map,SEQNO_map,_PSU_map,CTELENM1_map,...,_RFSEAT2_map,_RFSEAT3_map,_DRNKDRV_map,LNDSXBRT,CELSXBRT,BIRTHSEX,TRNSGNDR,USEMRJN4,RCSGEND1,RCSXBRTH
0,Alabama,January,b'03012023',b'03',b'01',b'2023',Completed Interview,b'2023000001',2023000001,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
1,Alabama,January,b'01062023',b'01',b'06',b'2023',Completed Interview,b'2023000002',2023000002,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
2,Alabama,January,b'03082023',b'03',b'08',b'2023',Completed Interview,b'2023000003',2023000003,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
3,Alabama,January,b'03062023',b'03',b'06',b'2023',Completed Interview,b'2023000004',2023000004,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
4,Alabama,January,b'01062023',b'01',b'06',b'2023',Completed Interview,b'2023000005',2023000005,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Have not driven after having too much to drink,NaN,NaN,NaN,4.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,Virgin Islands,December,b'12112023',b'12',b'11',b'2023',Completed Interview,b'2023002064',2023002064,MissingNotes: QSTVER > = 20,...,Always or Almost Always Wear Seat Belt,Don't Always Wear Seat Belt,Have not driven after having too much to drink,NaN,NaN,NaN,4.0,NaN,NaN,NaN
433319,Virgin Islands,December,b'01032024',b'01',b'03',b'2024',Completed Interview,b'2023002065',2023002065,MissingNotes: QSTVER > = 20,...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
433320,Virgin Islands,December,b'12132023',b'12',b'13',b'2023',Completed Interview,b'2023002066',2023002066,MissingNotes: QSTVER > = 20,...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Have not driven after having too much to drink,NaN,NaN,NaN,4.0,NaN,NaN,NaN
433321,Virgin Islands,December,b'12082023',b'12',b'08',b'2023',Completed Interview,b'2023002067',2023002067,MissingNotes: QSTVER > = 20,...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN


## 2 Dado tratado 




In [ ]:
'''   
Certamente as colunas relacionadas a identificação da amostra
relacionadas a datas por exemplo podem ser retiradas



sem os indices que correspondem a strings que podem ser nulas como 
Don’t know/Not sure and Refused to answer,
- deixar vazio e imputar zero 
- se for nas variáveis de interesse remover o dado 


Normalizar valores contínuos



'''

Retirar do dado os valores que representam strings indesejadas no codebook

In [ ]:
# Quando o valor no codebook for algum desses da lista , mudaremos para vazio
df_remove_string_index = df_2023.copy()
string_list_to_null = ["Refused", "Don’t know", "Not sure", "Don’t know/Not sure", "Refused to answer", "Blank", "Missing", "Not asked or Missing", "None"]
# df_withhout_invalid = limpar_valores_indesejados_codebook(df_remove_string_index, codebook_2023, string_list_to_null)
# alteramos o argumento para substituir por vazio
df_withhout_invalid = substituir_valores_por_zero_baseado_no_codebook(df_remove_string_index, codebook_2023, string_list_to_null, sufixo='_processed')
df_withhout_invalid.to_csv(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023_string_null.csv", index=False)
df_withhout_invalid


Processando colunas:  29%|██▉       | 101/350 [02:18<09:09,  2.21s/it]/tmp/ipykernel_7484/2967148062.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_processado[coluna + sufixo] = serie_modificada
Processando colunas:  29%|██▉       | 102/350 [02:19<07:14,  1.75s/it]/tmp/ipykernel_7484/2967148062.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_processado[coluna + sufixo] = serie_modificada
Processando colunas:  29%|██▉       | 103/350 [02:22<08:15,  2.01s/it]/tmp/ipykernel_7484/2967148062.py:78: PerformanceWarning

,LNDSXBRT,CELSXBRT,BIRTHSEX,TRNSGNDR,USEMRJN4,RCSGEND1,RCSXBRTH,_STATE_processed,FMONTH_processed,IDATE_processed,...,DROCDY4__processed,_RFBING6_processed,_DRNKWK2_processed,_RFDRHV8_processed,_FLSHOT7_processed,_PNEUMO3_processed,_AIDTST4_processed,_RFSEAT2_processed,_RFSEAT3_processed,_DRNKDRV_processed
0,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'03012023',...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,NaN
1,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'01062023',...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,NaN
2,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'03082023',...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,NaN
3,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'03062023',...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,1.0,1.0,1.0,NaN
4,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'01062023',...,7.000000e+00,1.0,4.700000e+01,1.0,2.0,1.0,2.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,NaN,NaN,NaN,4.0,NaN,NaN,NaN,78.0,12.0,b'12112023',...,7.100000e+01,2.0,1.500000e+03,2.0,2.0,2.0,1.0,1.0,2.0,2.0
433319,NaN,NaN,NaN,4.0,NaN,NaN,NaN,78.0,12.0,b'01032024',...,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,1.0,1.0,1.0,NaN
433320,NaN,NaN,NaN,4.0,NaN,NaN,NaN,78.0,12.0,b'12132023',...,3.000000e+00,1.0,4.700000e+01,1.0,NaN,NaN,1.0,1.0,1.0,2.0
433321,NaN,NaN,NaN,4.0,NaN,NaN,NaN,78.0,12.0,b'12082023',...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,NaN


In [ ]:
'''
Recaptulando
Variáveis de interesse 
CVDINFR4 = Ever Diagnosed with Heart Attack 
CVDCRHD4 = Ever Diagnosed with Angina or Coronary Heart Disease 
_MICHD = Ever Diagnosed with Heart Disease 

'''

# assim vamos retirar os registros (linhas) onde é vazio para a variável MICHD
# removar o missing data por linha , todos que foem nulos para as variável MICHD

df_remove_null_lines = df_withhout_invalid.copy()
df_remove_null_lines_result = remover_linhas_com_alvo_nulo(df_remove_null_lines, '_MICHD_processed')
df_remove_null_lines_result.to_csv(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023_michd_not_null.csv", index=False)
df_remove_null_lines_result


Coluna alvo para remoção de nulos: '_MICHD_processed'
Linhas antes da remoção: 433323
Linhas removidas: 4585
Linhas após a remoção: 428738


,LNDSXBRT,CELSXBRT,BIRTHSEX,TRNSGNDR,USEMRJN4,RCSGEND1,RCSXBRTH,_STATE_processed,FMONTH_processed,IDATE_processed,...,DROCDY4__processed,_RFBING6_processed,_DRNKWK2_processed,_RFDRHV8_processed,_FLSHOT7_processed,_PNEUMO3_processed,_AIDTST4_processed,_RFSEAT2_processed,_RFSEAT3_processed,_DRNKDRV_processed
0,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'03012023',...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,NaN
1,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'01062023',...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,NaN
2,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'03082023',...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,NaN
3,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'03062023',...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,1.0,1.0,1.0,NaN
4,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1.0,1.0,b'01062023',...,7.000000e+00,1.0,4.700000e+01,1.0,2.0,1.0,2.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,NaN,NaN,NaN,4.0,NaN,NaN,NaN,78.0,12.0,b'12112023',...,7.100000e+01,2.0,1.500000e+03,2.0,2.0,2.0,1.0,1.0,2.0,2.0
433319,NaN,NaN,NaN,4.0,NaN,NaN,NaN,78.0,12.0,b'01032024',...,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,1.0,1.0,1.0,NaN
433320,NaN,NaN,NaN,4.0,NaN,NaN,NaN,78.0,12.0,b'12132023',...,3.000000e+00,1.0,4.700000e+01,1.0,NaN,NaN,1.0,1.0,1.0,2.0
433321,NaN,NaN,NaN,4.0,NaN,NaN,NaN,78.0,12.0,b'12082023',...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,NaN


Tratamento de variáveis contínuas

Outros tratamentos de strings e dados de data

Tratamento de missing data (30% conforme o paper)

In [ ]:

# a amostra onde a linha tiver mais de 30% de missing data


In [24]:
# Aplicar seleção de variáveis com menos de 25 % de missing após filtrar o Heart Attack and Stroke
# df_brfss_2023_filtered_columns_csv = pd.read_csv(f"{notebook_dir_project_predict}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023.csv")
df_brfss_2023_filtered_columns_csv = df_withhout_invalid.copy()

# Calcular o percentual de valores nulos por coluna
percentual_na = df_brfss_2023_filtered_columns_csv.isnull().mean() * 100

# Selecionar colunas com menos de 30% de valores nulos - usar < 30
colunas_boas = percentual_na[percentual_na  < 10].index # Usar a proporção de missing data da variavel alvo para filtrar as colunas

# Criar novo DataFrame apenas com essas colunas
df_brfss_2023_csv_filtered = df_brfss_2023_filtered_columns_csv[colunas_boas]

# Exibir informações do DataFrame filtrado
df_brfss_2023_csv_filtered.info()

# Exibir o DataFrame
df_brfss_2023_csv_filtered


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 433323 entries, 0 to 433322
Columns: 111 entries, _STATE_processed to _RFSEAT3_processed
dtypes: float64(106), object(5)
memory usage: 367.0+ MB


,_STATE_processed,FMONTH_processed,IDATE_processed,IMONTH_processed,IDAY_processed,IYEAR_processed,DISPCODE_processed,SEQNO_processed,_PSU_processed,SEXVAR_processed,...,_SMOKER3_processed,_RFSMOK3_processed,_CURECI2_processed,DRNKANY6_processed,DROCDY4__processed,_RFBING6_processed,_DRNKWK2_processed,_RFDRHV8_processed,_RFSEAT2_processed,_RFSEAT3_processed
0,1.0,1.0,b'03012023',b'03',b'01',b'2023',1100.0,b'2023000001',2.023000e+09,2.0,...,4.0,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0
1,1.0,1.0,b'01062023',b'01',b'06',b'2023',1100.0,b'2023000002',2.023000e+09,2.0,...,4.0,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0
2,1.0,1.0,b'03082023',b'03',b'08',b'2023',1100.0,b'2023000003',2.023000e+09,2.0,...,3.0,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0
3,1.0,1.0,b'03062023',b'03',b'06',b'2023',1100.0,b'2023000004',2.023000e+09,2.0,...,4.0,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0
4,1.0,1.0,b'01062023',b'01',b'06',b'2023',1100.0,b'2023000005',2.023000e+09,2.0,...,4.0,1.0,1.0,1.0,7.000000e+00,1.0,4.700000e+01,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,78.0,12.0,b'12112023',b'12',b'11',b'2023',1100.0,b'2023002064',2.023002e+09,1.0,...,4.0,1.0,1.0,1.0,7.100000e+01,2.0,1.500000e+03,2.0,1.0,2.0
433319,78.0,12.0,b'01032024',b'01',b'03',b'2024',1100.0,b'2023002065',2.023002e+09,2.0,...,4.0,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0
433320,78.0,12.0,b'12132023',b'12',b'13',b'2023',1100.0,b'2023002066',2.023002e+09,2.0,...,4.0,1.0,1.0,1.0,3.000000e+00,1.0,4.700000e+01,1.0,1.0,1.0
433321,78.0,12.0,b'12082023',b'12',b'08',b'2023',1100.0,b'2023002067',2.023002e+09,2.0,...,4.0,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0


In [ ]:
# ver libs de imputação de dados com oknn e cat boost

Dataset final tratado para uso nos modelos